In [1]:
import pandas as pd
import re
import ast

# Training completions

In [ ]:
data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_filtered.csv") 
# data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/with_summary/test_np.csv")

In [2]:
df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/logs/train_completions/qwen25_3B_drgrpo_gp32.csv")

/scratch/5766238.1.cds-gpu-long/ipykernel_3725509/3222431863.py:1: DtypeWarning: Columns (0,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/logs/train_completions/qwen25_3B_drgrpo_gp32.csv")


In [3]:
df = df[df['step'] != 'step']
df['step'] = df['step'].astype(int)
# df['hybrid_reward'] = df['hybrid_rewar/d'].astype(float)
df['correctness_reward'] = df['correctness_reward'].astype(float)
df['format_reward'] = df['format_reward'].astype(float)
df['tag_count_reward'] = df['tag_count_reward'].astype(float)
df['advantage'] = df['advantage'].astype(float)
df = df.drop_duplicates(subset=["prompt", "completion"]).reset_index(drop=True)

In [4]:
print(len(df['ID'].unique()))
print(len(df['prompt'].unique()))

43040
43040


In [5]:
df.head()

,step,ID,ground_truth,ground_truth_text,prompt,completion,correctness_reward,format_reward,tag_count_reward,advantage
0,1,NACC943915,A,Normal Cognition (NC),<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, let's summarize the patient's ...",1.0,1.0,0.5,-0.007812
1,1,NACC943915,A,Normal Cognition (NC),<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, let's summarize the patient's ...",1.0,1.0,1.0,0.492188
2,1,NACC943915,A,Normal Cognition (NC),<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, I will provide a general summa...",0.0,1.0,1.0,-0.507812
3,1,NACC943915,A,Normal Cognition (NC),<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, let's summarize the patient's ...",1.0,1.0,1.0,0.492188
4,1,NACC943915,A,Normal Cognition (NC),<|im_start|>system\nYou are a helpful AI assis...,"<think>\nFirst, I will summarize the patient's...",1.0,1.0,1.0,0.492188


In [6]:
print(df.iloc[-1]['completion'])

<think>
First, let's summarize the patient's presentation:
- The patient is a 58-year-old female with multiple comorbid conditions: hypertension, hypercholesterolemia, sleep apnea.
- Medications include treatments for Alzheimer’s (memantine and donepezil), cognitive enhancers (cognitive behavioral therapy substance, ubiquinone), lipid lowerer (atorvastatin), and other neuroprotective agents.
- Co-occurring psychiatric symptoms include: mild disinhibition and mild nighttime behaviors.
- Physical examination: no signs of Parkinson’s disease, cerebrovascular disease, or abnormal vision/hearing.
- Cognitive assessment suggests mild impairment but normal initial findings, including variability in performance tasks due to cognitive or behavioral issues. 
- Key neuropsychological findings: variability, especially in tasks related to visual-spatial and executive functions, complete failure due to cognitive or behavioral issues, and a MoCA score of 3 out of 32.
Given the focusing on non-invasiv

In [7]:
len(df)

732160

In [7]:
# df['reward'] = df['hybrid_reward'] + df['format_reward'] + df['tag_count_reward']
df['reward'] = df['correctness_reward'] + df['format_reward'] + df['tag_count_reward']

In [8]:
completion_df = pd.DataFrame(df['completion'].value_counts()).reset_index()

In [9]:
completion_df[completion_df['count'] > 1]

,completion,count


In [10]:
# print(completion_df[completion_df['count'] > 1].iloc[-1]['completion'])

In [11]:
len(df['ID'].unique())

43040

In [12]:
df['advantage_std'] = df.groupby('ID')['advantage'].transform('std')
len(df[df['advantage_std'] == 0.0].groupby('ID')['advantage'].value_counts().reset_index(name='count'))

28044

In [13]:
# df[df['advantage'] == 0].groupby('prompt')['correctness_reward'].value_counts().reset_index(name='count')['correctness_reward'].value_counts()
# sub_df = df[df['reward_std'] == 0.0].groupby('prompt')['reward'].value_counts().reset_index(name='count')
df['reward_std'] = df.groupby('ID')['reward'].transform('std')
sub_df = df[df['reward_std'] == 0.0].groupby('ID')['reward'].value_counts().reset_index(name='count')
print(len(sub_df))
sub_df['reward'].value_counts()

28044


reward
3.0    22443
2.0     5601
Name: count, dtype: int64

In [23]:
for i, row in df.iterrows():
    if "Alzheimer's disease pathology (AD) only" in row['prompt']:
        print(i)
        break

608


In [24]:
index = 608
print(df.iloc[index]['ID'])
print(df.iloc[index]['prompt'])
print(df.iloc[index]['ground_truth_text'])
# print(df.iloc[index]['ground_truth'], data[data['ID'] == df.iloc[index]['ID']].iloc[0]['ground_truth_text'])
print(df.iloc[index]['completion'])
print(df.iloc[index]['correctness_reward'])

NACC298369
<|im_start|>system
You are a helpful AI assistant that provides well-reasoned and detailed responses. First, think step by step through the reasoning process as an internal monologue, and then provide the user with the final answer within \boxed{}. Respond using the following format: <think>
...
</think>
<answer>
...
</answer>, i.e., the reasoning process should be enclosed within <think>...</think> tags, and the final answer within <answer>...</answer> tags. Reply in English only, do not use other languages.<|im_end|>
<|im_start|>user
Below is the background of the patient.
<patient_data>
The subject is a 92-year-old female who presented at the Alzheimer's Disease Center (ADC) to participate in a research study. She was referred by a professional contact, which may include a clinician, nurse, doctor, or other healthcare provider. The subject is White, non-Hispanic/Latino, and her primary language is English. She has completed 16 years of education and is married, currently 

In [17]:
# check reflection patterns
cnt = []
for i, row in df.iterrows():
    if row['completion'].lower().count("but wait,") >= 1:
        # print("The word 'wait' appears at least twice.", i)
        # print(row['completion'])
        # break
        cnt.append(i)
        
len(cnt)

0

In [34]:
print(df.iloc[cnt[-1]]['ground_truth'])
print(df.iloc[cnt[-1]]['completion'])

B
<think>
First, let's analyze the patient's clinical presentation. Key features indicates significant cognitive impairment, motor dysfunction, and behavioral symptoms. The patient has a recent diagnosis of Parkinson's disease, which is a neurodegenerative disorder affecting the substantia nigra region of the brain. Several important symptoms have been identified, including urinary and bowel incontinence, severe rigidity and bradykinesia across the upper and lower extremities, and a characteristic masked facial expression and inability to speak coherently. 

Upon examination, the patient's neuropsychological testing revealed an extremely low MMSE score of 6. Given his tests' results, such as significant deficits in memory tasks, delayed executive functioning, and performance on visual-spatial tasks, it's evident that his cognitive functions are severely impaired.

Examining the provided options, we should narrow this down to one of the causes that would explain these symptoms:

A. Is i

# Inference

In [181]:
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_mci/2025-06-07T195120_841fe23217cc449d/test_mci_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_mci/2025-06-07T200518_ab1e47eb4d6e4207/test_mci_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_np/2025-05-26T013154_fadcfac71d954c1b/test_np_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_np/2025-06-01T154608_8acf8fe2d7234c55/test_np_output.jsonl"
# result_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/results_3/test_np/2025-05-22T102606_e437f90d9d214379/test_np_output.jsonl"
# df = pd.read_json(result_path, lines=True)

In [93]:
df = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/extracted_results_4/PRION/qwen25_3B_sft_PRION.csv")

In [94]:
len(df)

1780

In [95]:
sub_df = df[df['extraction_type'] == "llm"]

In [96]:
len(sub_df)

1

In [97]:
df['prediction'].value_counts()

prediction
B    1300
A     480
Name: count, dtype: int64

In [98]:
index = 10
print(df.iloc[index]['ground_truth'])
print(df.iloc[index]['prediction'])
print(df.iloc[index]['generated_text'])
print("visit_summary: ")
print(ast.literal_eval(df.iloc[index]['problem'])['visit_summary'])
print(ast.literal_eval(df.iloc[index]['problem'])['options'])

B
B
The answer is B: No
visit_summary: 
The patient is a 76-year-old right-handed male who is married and lives with his spouse in a private residence. He is White, speaks English as his primary language, and has completed 16 years of education. He is currently independent in daily living and reports no cognitive concerns, having come to the Alzheimer’s Disease Center primarily to participate in a research study. There is no family history of cognitive impairment among first-degree relatives. The patient is taking three medications, including lisinopril, an ACE inhibitor, and also reports the use of an anxiolytic, sedative, or hypnotic agent, as well as an antihypertensive. He has a history of smoking, having smoked one and a half to two packs per day for 20 years before quitting at age 37. He has no history of traumatic brain injury, stroke, Parkinson’s disease, or other neurological conditions, but he does have active diagnoses of hypertension, diabetes, and thyroid disease.

Physica

In [12]:
generated_texts = list(df['generated_text'])

In [13]:
# generated_text = []
# for gen_text in generated_texts:
#     generated_text += gen_text

In [21]:
generated_text_df = pd.DataFrame(generated_texts, columns=['generated_text']).reset_index(drop=True)

In [22]:
completion_df = pd.DataFrame(generated_text_df['generated_text'].value_counts()).reset_index()

In [23]:
generated_text_df

,generated_text
0,"<think>\nFirst, let's summarize the patient's ..."
1,"<think>\nFirst, let's summarize the patient's ..."
2,"<think>\nFirst, let's provide a comprehensive ..."
3,"<think>\nFirst, let's summarize the patient's ..."
4,"<think>\nFirst, let's summarize the patient's ..."
...,...
20015,"<think>\nFirst, let's summarize the patient's ..."
20016,"<think>\nFirst, let's summarize the patient's ..."
20017,"<think>\nFirst, let's summarize the patient's ..."
20018,"<think>\nFirst, let's summarize the patient's ..."


In [24]:
completion_df[completion_df['count'] > 1]

,generated_text,count
0,"<think>\nFirst, let's summarize the patient's ...",31
1,"<think>\nFirst, let's summarize the patient's ...",13
2,"<think>\nFirst, let's summarize the patient's ...",10
3,"<think>\nFirst, let's provide a comprehensive ...",9
4,"<think>\nFirst, let's summarize the patient's ...",8
5,"<think>\nFirst, summarize the patient's presen...",7
6,"<think>\nFirst, let's summarize the patient's ...",6
7,"<think>\nFirst, let's summarize the patient's ...",6
8,"<think>\nFirst, let's summarize the patient's ...",6
9,"<think>\nFirst, let's summarize the patient's ...",5


In [25]:
completion_df[completion_df['count'] > 1].iloc[51]['generated_text']

"<think>\nFirst, let's summarize the patient's presentation based on the clinical information provided.\n</think>\n<answer>\nAnswer: H\n</answer>"

In [ ]:
from safetensors.torch import load_file
import torch

def safetensors_equal(file1, file2, rtol=1e-5, atol=1e-8):
    tensors1 = load_file(file1)
    tensors2 = load_file(file2)

    if tensors1.keys() != tensors2.keys():
        return False

    for key in tensors1.keys():
        t1 = tensors1[key]
        t2 = tensors2[key]
        
        # If types or shapes don't match, they're not equal
        if t1.dtype != t2.dtype or t1.shape != t2.shape:
            return False
        
        # For floating point, use allclose; for others, use exact match
        if torch.is_floating_point(t1):
            if not torch.allclose(t1, t2, rtol=rtol, atol=atol):
                return False
        else:
            if not torch.equal(t1, t2):
                return False

    return True

# Example usage
equal = safetensors_equal("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/ckpt_sft/qwen25_3B_sft_with_eval/checkpoint-500/model-00001-of-00002.safetensors", "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/open-r1/ckpt_sft/qwen25_3B_sft_with_eval/model-00001-of-00002.safetensors")
print("Equal:", equal)